# Photonic MZI: builder API, kfnetlist round trip, export

A Mach-Zehnder interferometer built hierarchically with the
`kfnetlist.morphnet` builder API. The notebook then:

1. converts it to `kfnetlist.Netlist` graphs,
2. flattens the coupler sub-cells with kfnetlist,
3. exports the flat netlist to Verilog-AMS, Spectre and SPICE.

Adapted from morphnet's `examples/mzi.py`. The changes:

- Each delay arm now sits *between* the couplers (`inp`, then `out`); the
  original wired the arm input and the next coupler to the same net.
- The external circuits declare their ports.
- Nets that reach a circuit terminal are named after it.
- Instance names start with `X`, SPICE's prefix for subcircuit calls, so the
  SPICE export is valid.

In [1]:
from kfnetlist import morphnet
from kfnetlist.morphnet.netlist import write_spectre, write_spice, write_vams

WG = morphnet.SignalDomain.WAVEGUIDE
IN, OUT = morphnet.PortDirection.INPUT, morphnet.PortDirection.OUTPUT

## Directional coupler sub-cell

Parameter defaults are given as `(name, value)` tuples. The terminals are in
the waveguide domain.

In [2]:
dc = morphnet.circuit("dc_coupler", parameters=[("gap", 200e-9), ("length", 10e-6)])
for name in ("i1", "i2", "o1", "o2"):
    dc.add_terminal(name, direction=IN if name[0] == "i" else OUT, domain=WG)

dc.add(morphnet.subcircuit("Xcoupler", "wg_directional_coupler"))
for name in ("i1", "i2", "o1", "o2"):
    dc.connect(name, ("", name), ("Xcoupler", name))

## MZI top cell

Two couplers, with a delay arm on each path between them.

In [3]:
mzi = morphnet.circuit("mzi", parameters=[("arm_length", 100e-6)])
for name in ("in1", "in2", "out1", "out2"):
    mzi.add_terminal(name, direction=IN if name.startswith("in") else OUT, domain=WG)

mzi.add(
    morphnet.subcircuit("Xdc1", "dc_coupler"),
    morphnet.subcircuit("Xdc2", "dc_coupler"),
    morphnet.subcircuit("Xarm_top", "wg_delay"),
    morphnet.subcircuit("Xarm_bot", "wg_delay"),
)
mzi.connect("in1", ("", "in1"), ("Xdc1", "i1"))
mzi.connect("in2", ("", "in2"), ("Xdc1", "i2"))
mzi.connect("top_a", ("Xdc1", "o1"), ("Xarm_top", "inp"))
mzi.connect("top_b", ("Xarm_top", "out"), ("Xdc2", "i1"))
mzi.connect("bot_a", ("Xdc1", "o2"), ("Xarm_bot", "inp"))
mzi.connect("bot_b", ("Xarm_bot", "out"), ("Xdc2", "i2"))
mzi.connect("out1", ("Xdc2", "o1"), ("", "out1"))
mzi.connect("out2", ("Xdc2", "o2"), ("", "out2"))

Circuit(name='mzi', class_name='', ports=[Port(name='in1', direction=<PortDirection.INPUT: 1>, domain=<SignalDomain.WAVEGUIDE: 2>, width=0, properties={}, discipline=''), Port(name='in2', direction=<PortDirection.INPUT: 1>, domain=<SignalDomain.WAVEGUIDE: 2>, width=0, properties={}, discipline=''), Port(name='out1', direction=<PortDirection.OUTPUT: 2>, domain=<SignalDomain.WAVEGUIDE: 2>, width=0, properties={}, discipline=''), Port(name='out2', direction=<PortDirection.OUTPUT: 2>, domain=<SignalDomain.WAVEGUIDE: 2>, width=0, properties={}, discipline='')], parameters=[Parameter(name='arm_length', default_value=ParameterValue(prefixed_value=PrefixedValue(double_value=0.0001, prefix=<SIPrefix.UNSPECIFIED: 0>), model_ref=None), description='', properties={})], model_interfaces=[], subcircuits=[SubCircuit(name='Xdc1', circuit_name='dc_coupler', class_name='', parameters=[], properties={}), SubCircuit(name='Xdc2', circuit_name='dc_coupler', class_name='', parameters=[], properties={}), SubC

## Netlist with a photonic PDK library

In [4]:
ckt = morphnet.Netlist(name="mzi_chip", top_circuit="mzi")
ckt.add_library(
    "photonic_pdk",
    morphnet.external_library(
        "path/to/photonic_pdk.lib",
        model_type="photonic",
        external_model_names=["wg_directional_coupler", "wg_delay"],
    ),
)
ckt.add_ext(
    morphnet.ext_circuit(
        "wg_directional_coupler",
        class_name="dc",
        library_name="photonic_pdk",
        ports=[morphnet.port(p, domain=WG) for p in ("i1", "i2", "o1", "o2")],
    ),
    morphnet.ext_circuit(
        "wg_delay",
        class_name="delay_line",
        library_name="photonic_pdk",
        ports=[morphnet.port(p, domain=WG) for p in ("inp", "out")],
    ),
)
ckt.add(mzi, dc)

mzi_module = next(m for m in ckt.circuits if m.name == "mzi")
assert all(t.domain == WG for t in mzi_module.ports)
print(write_vams(ckt))

module mzi (in1, in2, out1, out2);
  input in1, in2;
  output out1, out2;
  parameter arm_length = 0.0001;
  dc_coupler Xdc1 (.i1(in1), .i2(in2), .o1(top_a), .o2(bot_a));
  dc_coupler Xdc2 (.i1(top_b), .i2(bot_b), .o1(out1), .o2(out2));
  wg_delay Xarm_top (.inp(top_a), .out(top_b));
  wg_delay Xarm_bot (.inp(bot_a), .out(bot_b));
endmodule
module dc_coupler (i1, i2, o1, o2);
  input i1, i2;
  output o1, o2;
  parameter gap = 2e-07;
  parameter length = 1e-05;
  wg_directional_coupler Xcoupler (.i1(i1), .i2(i2), .o1(o1), .o2(o2));
endmodule



## Into kfnetlist

`to_netlists` gives one `kfnetlist.Netlist` per circuit, so all of kfnetlist's
tooling applies: open detection, LVS diffs, flattening.

In [5]:
netlists = morphnet.to_netlists(ckt)
print({name: nl for name, nl in netlists.items()})
print("opens:", netlists["mzi"].detect_opens())

{'mzi': Netlist(instances=4, nets=8, ports=4), 'dc_coupler': Netlist(instances=1, nets=4, ports=4)}
opens: {'unconnected_ports': [], 'singleton_nets': []}


Flatten the two coupler instances into their contents:

In [6]:
flat = netlists["mzi"].flatten(
    netlists,
    instance_cell_map={"Xdc1": "dc_coupler", "Xdc2": "dc_coupler"},
    separator="_",  # keeps flattened names valid identifiers in every format
)
flat.sort()
print("instances:", flat.instance_names())

instances: ['Xarm_bot', 'Xarm_top', 'Xdc1_Xcoupler', 'Xdc2_Xcoupler']


## And back out, in any format

`from_netlists` turns the flat `kfnetlist.Netlist` back into a morphnet `Netlist`. The
external circuits are declared again so the positional formats keep the pin order.

In [7]:
flat_ckt = morphnet.from_netlists({"mzi_flat": flat}, "mzi_flat")
flat_ckt.add_ext(*ckt.ext_circuits)
print(write_vams(flat_ckt))
print(write_spectre(flat_ckt))
print(write_spice(flat_ckt))

module mzi_flat (in1, in2, out1, out2);
  inout in1, in2, out1, out2;
  wg_delay Xarm_bot (.inp(net_4), .out(net_5));
  wg_delay Xarm_top (.inp(net_6), .out(net_7));
  wg_directional_coupler Xdc1_Xcoupler (.i1(in1), .i2(in2), .o1(net_6), .o2(net_4));
  wg_directional_coupler Xdc2_Xcoupler (.i1(net_7), .i2(net_5), .o1(out1), .o2(out2));
endmodule

subckt mzi_flat (in1 in2 out1 out2)
Xarm_bot (net_4 net_5) wg_delay
Xarm_top (net_6 net_7) wg_delay
Xdc1_Xcoupler (in1 in2 net_6 net_4) wg_directional_coupler
Xdc2_Xcoupler (net_7 net_5 out1 out2) wg_directional_coupler
ends mzi_flat

* mzi_flat
.subckt mzi_flat in1 in2 out1 out2
Xarm_bot net_4 net_5 wg_delay
Xarm_top net_6 net_7 wg_delay
Xdc1_Xcoupler in1 in2 net_6 net_4 wg_directional_coupler
Xdc2_Xcoupler net_7 net_5 out1 out2 wg_directional_coupler
.ends mzi_flat
.end

